In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("digital_footprints.csv", sep=";")

In [2]:
df.head()

,client_id,visitor_id,visit_id,process_step,date_time,date,time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,2017-04-17,15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,2017-04-17,15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,2017-04-17,15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,2017-04-17,15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,2017-04-17,15:18:04


In [3]:
# drop nulls
df = df.dropna(subset=['process_step'])

In [4]:
#Extract the process step number (e.g., 'step_3' -> 3)
df['step_number'] = df['process_step'].str.extract(r'(\d+)').astype(float)

In [5]:
#Maximun number of steps
max_step = int(df['step_number'].max())

In [6]:
# Group by client_id and find the highest step reached by each client
completion_df = df.groupby('client_id')['step_number'].max().reset_index()

In [7]:
# Check if the customer reached the final step
completion_df['completed'] = completion_df['step_number'] == max_step

In [8]:
# Calculate the completion rate (proportion of customers who completed the process)
completion_rate = completion_df['completed'].mean()

In [9]:
print(f"Completion rate per customer: {completion_rate:.2%}")

Completion rate per customer: 75.80%


In [10]:
# OTHER WAY  -----------------Using confirm as the value -----------------------------

confirmed_df = df.groupby('client_id')['process_step'].apply(lambda x: 'confirm' in x.values).reset_index()
confirmed_df.rename(columns={'process_step': 'completed'}, inplace=True)

In [11]:
completion_rate = confirmed_df['completed'].mean()

In [12]:
print(f"Completion rate per customer: {completion_rate:.2%}")

Completion rate per customer: 67.53%


In [13]:
df1 = pd.read_csv("clean_final_experiment.csv")
df1 = df1.dropna()
df1.head()


,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control


In [14]:
# Merge both tables by client_id
df = pd.merge(df, df1, on='client_id', how='inner')


In [29]:
# Create a column indicating the group (new or old layout)
df['group'] = df['Variation'].apply(lambda x: 'test' if x == 'Test' else 'control')

In [30]:
df.columns


Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'date', 'time', 'step_number', 'Variation', 'group', 'completed'],
      dtype='object')

In [31]:
from scipy.stats import norm
import numpy as np

# Create a boolean column 'completed' if the user reached the "confirm" stage
df['completed'] = df['process_step'].str.lower() == 'confirm'

# Group by user to keep in mind if they completed the process (at least once)
user_completion = df.groupby(['client_id', 'group'])['completed'].max().reset_index()

# Count completed and total by group
user_completion = user_completion.dropna(subset=['group', 'completed'])

summary = (
    user_completion
    .groupby('group')['completed']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'completions', 'count': 'total'})
)
print(summary)



         completions  total
group                      
control        15434  23532
test           18687  26968


In [18]:

# Get values
completions = summary['completions'].values
totals = summary['total'].values

# Proportions
p1 = completions[0] / totals[0]
p2 = completions[1] / totals[1]

In [19]:
# Combined ratio
p_pool = (completions[0] + completions[1]) / (totals[0] + totals[1])

In [20]:
#standar error
se = np.sqrt(p_pool * (1 - p_pool) * (1/totals[0] + 1/totals[1]))

In [21]:
# Estadístico z
z = (p1 - p2) / se

In [22]:
# two-sided p-value
p_value = 2 * (1 - norm.cdf(abs(z)))

In [23]:
# Show results
print(f"\nZ-statistic: {z:.4f}")
print(f"P-value: {p_value:.4f}")


Z-statistic: 8.8745
P-value: 0.0000


In [24]:
if p_value < 0.05:
    print("The difference in conversion rates is statistically significant.")
else:
    print(" There is insufficient evidence to say that the difference is significant.")

The difference in conversion rates is statistically significant.


In [27]:
completion_rate_old = summary.loc['old', 'completions'] / summary.loc['old', 'total']
print(f"Completion rate (control): {completion_rate_old:.2%}")



Completion rate (control): 65.59%


In [32]:
completion_rates = summary['completions'] / summary['total']
print(completion_rates.apply(lambda x: f"{x:.2%}"))

group
control    65.59%
test       69.29%
dtype: object
